
# 第 10 节：深度 Q 网络（DQN）- 从零实现

---

## 课程位置

```
Tabular Q-Learning (08) -> 函数逼近 (09) -> **DQN (10)** -> Policy Gradient (11) -> ...
                                                  ^
                                             你在这里
```

DQN 是深度强化学习**第一个**里程碑式的工作（Mnih et al., 2013, 2015），首次将深度神经网络与 Q-Learning 结合，
在 Atari 游戏上达到了超越人类玩家的水平。本节将从零开始实现 DQN，并在 CartPole 环境中验证其效果。



## 学习目标

完成本节后，你将能够：

1. **理解**：为什么将 Q-Learning 与神经网络直接结合会不稳定
2. **掌握**：DQN 的三大核心创新 - 经验回放、目标网络、Huber 损失
3. **实现**：从零编写完整的 DQN 训练流程
4. **分析**：Q 值变化趋势、epsilon 衰减曲线、损失曲线
5. **消融**：分别移除目标网络和经验回放，观察训练崩溃现象
6. **应用**：将训练好的 DQN 智能体录制为视频



## 1. DQN 动机：为什么 Q-Learning + 神经网络很难？

### 回顾：Q-Learning 更新公式

**Tabular Q-Learning** 的更新规则：

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t) \right]
$$

在表格方法中，每个 (s, a) 对独立存储和更新，互不干扰。

### 函数逼近的挑战

当我们用神经网络 Q(s, a; theta) 代替 Q 表时，出现了三个关键问题：

| 问题 | 描述 | 后果 |
|------|------|------|
| **数据相关性** | 连续 step 的 (s, a, r, s') 高度相关，不满足 i.i.d. 假设 | 梯度更新方差大，参数震荡 |
| **目标非平稳** | TD target r + gamma max Q(s', a'; theta) 依赖当前 theta，每次更新都改变目标 | 类似"追自己的尾巴"，难以收敛 |
| **梯度尺度** | TD error 可能很大，导致梯度爆炸 | 训练不稳定，loss 发散到 NaN |

### DQN 的三大创新

2013 年 DeepMind 的 DQN 通过以下三项关键技术解决了上述问题：

1. **Experience Replay（经验回放）** -> 解决数据相关性
2. **Target Network（目标网络）** -> 解决目标非平稳
3. **Reward Clipping / Huber Loss** -> 解决梯度尺度

下面逐一详解。



## 2. 创新一：Experience Replay（经验回放）

### 直觉

想象你是一位学生：
- **在线学习**（Online）：每做一道题就立刻学习，题目顺序高度相关 -> 学完容易忘
- **经验回放**：把做过的题目都记录在错题本上，随机抽取复习 -> 知识更牢固

### 工作原理

1. **存储**：将每一步的 transition (s_t, a_t, r_t, s_{t+1}, done_t) 存入循环缓冲区
2. **采样**：每次更新时，从缓冲区中**均匀随机采样**一个 minibatch
3. **学习**：用这个随机批次计算梯度并更新网络

### 为什么有效？

| 角度 | 在线学习 | 经验回放 |
|------|----------|----------|
| 相关性 | 连续样本高度相关 -> 梯度有偏 | 打破时序相关 -> 梯度近似 i.i.d. |
| 样本效率 | 用完即弃 | 每个经验可复用多次 |
| 数据分布 | 受当前策略影响 -> 分布偏移严重 | 混合历史经验 -> 分布更平滑 |

### 数学理解

在线学习时，每一次梯度更新使用的样本来自当前策略 pi_{theta_t} 的近期轨迹，
这些样本的联合分布 P(s_t, a_t, r_t, s_{t+1}) 高度偏离稳态分布。

经验回放使用一个**经验分布** D，其中每个 transition 被等概率采样：

$$
L(theta) = E_{(s, a, r, s', d) ~ D} [ (r + gamma max_{a'} Q_target(s', a') - Q(s, a; theta))^2 ]
$$

这使得梯度更新更稳定、更高效。

> **缓冲区大小**是重要超参数：
> - 太小 -> 无法覆盖足够多样的经验 -> 过拟合近期经验
> - 太大 -> 包含太多陈旧经验 -> 学习缓慢（策略已变但数据还在过去）



## 3. 创新二：Target Network（目标网络）

### 问题：目标非平稳

在标准 Q-Learning 中，TD target 为：

$$y_t = r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a'; \theta_t)$$

注意：**计算 target 和预测 Q 值使用了同一组参数 theta_t**。

这导致了一个矛盾：
- 我们希望 Q(s, a; theta) 逼近 y
- 但每次更新 theta，y 也随之改变
- 就像"狗追自己的尾巴"，目标一直在移动

### 解决方案：冻结目标网络

使用**两套 Q 网络**：

| 网络 | 角色 | 更新方式 |
|------|------|----------|
| **Online 网络** Q(s, a; theta) | 当前 Q 值预测、动作选择 | 每步通过梯度下降更新 |
| **Target 网络** Q(s, a; theta-) | 计算 TD target | 每隔 C 步从 online 网络**硬复制** |

### 更新后的目标

$$y_t = r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a'; \theta^-_t)$$

其中 theta- 在 C 步内保持不变，极大减少了目标波动。

### 效果分析

- **稳定性**：target 固定 -> 损失函数的优化 landscape 在一个周期内稳定
- **代价**：target 稍微滞后（stale），但这是稳定性和效率之间的必要权衡

> **Target update frequency C** 是重要超参数：
> - 太小（如 C=1）-> 退化为无 target network，训练容易发散
> - 太大（如 C=10000）-> target 过于陈旧，学习速度变慢
> - 常用值：CartPole 中 C=100，Atari 中 C=10000



## 4. 创新三：Huber Loss 与 Gradient Clipping

### 问题：梯度爆炸

Q-Learning 的 TD error 可能非常大，特别是在训练初期：
$$\delta = r + \gamma \max Q(s') - Q(s)$$

使用 MSE 损失时，梯度与误差成正比 nabla L = 2 * delta * nabla Q。
如果 |delta| 很大（如 100+），梯度会爆炸，导致网络参数剧烈震荡。

### 解决方案 1：Huber Loss

Huber Loss 结合了 MSE 和 MAE 的优点：

$$
L_{\delta}(x) = \begin{cases}
\frac{1}{2} x^2 & \text{if } |x| \leq \delta \\
\delta(|x| - \frac{1}{2}\delta) & \text{otherwise}
\end{cases}
$$

也称为 Smooth L1 Loss（PyTorch 中的 nn.SmoothL1Loss）。

| 范围 | 梯度行为 | 效果 |
|------|----------|------|
| |x| <= delta | 线性增长（同 MSE） | 小误差时精细调整 |
| |x| > delta | 常数（梯度 = +/- delta） | 大误差时防止梯度爆炸 |

### 解决方案 2：Gradient Clipping

即使使用 Huber Loss，**梯度裁剪**仍然是重要的安全网：

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
```

将梯度的 L2 范数限制在 max_norm 以内，防止罕见的极端情况。

### 解决方案 3：Reward Clipping（论文中使用）

在 Atari DQN 论文中，所有正奖励裁剪为 +1，负奖励裁剪为 -1。
这样做的好处：
- 统一不同游戏的奖励尺度
- 防止大奖励主导学习
- 但会丢失奖励幅度信息

> 在 CartPole 中奖励天然为 +1（每活一步），不存在尺度问题，因此我们不需要 reward clipping，但 Huber Loss 仍然是安全的默认选择。



## 5. DQN 算法伪代码

```
Algorithm: Deep Q-Network (DQN)
-------------------------------------------------------------------------
  1: 初始化 Q 网络 Q(s, a; theta)（随机权重）
  2: 初始化目标网络 Q(s, a; theta-) <- theta
  3: 初始化经验回放缓冲区 D（容量 N）
  4: 初始化探索率 eps = eps_start
  5:
  6: for episode = 1 to M do
  7:     获取初始状态 s_1
  8:     for t = 1 to T do
  9:         // eps-贪心动作选择
 10:         a = argmax Q(s, a; theta)    以概率 (1-eps)
 11:           = 随机动作                   以概率 eps
 12:
 13:         执行 a，获得 r, s', done
 14:
 15:         // 存入经验回放
 16:         D.push(s, a, r, s', done)
 17:
 18:         if len(D) >= batch_size then
 19:             // 采样训练
 20:             (s, a, r, s', d) ~ Uniform(D)
 21:
 22:             // 计算 TD target（使用目标网络）
 23:             y = r + gamma * max Q(s', a'; theta-) * (1 - d)
 24:
 25:             // 计算损失（MSE 或 Huber）
 26:             L = (y - Q(s, a; theta))^2
 27:
 28:             // 梯度下降
 29:             梯度下降 nabla_theta L
 30:
 31:             // 梯度裁剪（可选但推荐）
 32:             clip_grad_norm(theta, max_norm=10)
 33:
 34:         // 定期更新目标网络
 35:         if t % C == 0 then
 36:             theta- <- theta    // 硬复制
 37:
 38:         // 衰减探索率
 39:         eps = max(eps_end, eps * eps_decay)
 40:
 41:         if done then break
 42:     end for
 43: end for
-------------------------------------------------------------------------

> 核心公式回顾：
> $$L(theta) = E_{(s,a,r,s',d) ~ D} [ ( r + gamma * max_{a'} Q_target(s', a'; theta^-) * (1-d) - Q(s, a; theta) )^2 ]$$


In [ ]:
# ============================================================
# 环境准备：导入依赖、创建环境、固定随机种子
# ============================================================
import sys
import os
import math
import random
from collections import deque
from typing import List, Optional, Tuple, Dict
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib
matplotlib.use('Agg')  # 无头模式（服务器兼容）
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML
from tqdm import tqdm

# 课程模块中的可复用组件
from rl_course.utils.seeding import set_seed, get_device
from rl_course.networks.mlp import QNetwork
from rl_course.visualization.plotting import plot_learning_curve
from rl_course.visualization.video import record_episode

# ===== 创建输出目录 =====
os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/videos', exist_ok=True)

# ===== 固定全局随机种子 =====
set_seed(42)

# ===== 检测设备 =====
device = get_device()
print(f"设备: {device}")

# ===== 创建 CartPole 环境 =====
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"状态维度: {state_dim}  (位置, 速度, 角度, 角速度)")
print(f"动作空间: {n_actions}  (向左推, 向右推)")
print(f"目标: 保持杆子直立, 每活一步 +1 分, 上限 500 分")


In [ ]:
# ============================================================
# Q 网络：使用课程模块中的 QNetwork
# ============================================================
# QNetwork 来自 rl_course.networks.mlp.QNetwork
# 结构：输入(4) -> 隐藏层[64, 64] -> 输出(2)
#
# Q(s, a) 输出每个动作的 Q 值向量，形状 (batch_size, n_actions)
# 动作选择时取 argmax，梯度更新时 gather 实际执行动作的值

q_net_example = QNetwork(
    state_dim=state_dim,
    n_actions=n_actions,
    hidden_dims=[64, 64],
    activation='relu',
)

dummy_input = torch.randn(1, state_dim)
dummy_output = q_net_example(dummy_input)
print(f"Q 网络结构:")
print(f"  输入:  {state_dim} 维状态")
print(f"  隐藏层: [64, 64] ReLU + 正交初始化")
print(f"  输出:  {n_actions} 个动作的 Q 值")
print(f"  前向传播测试: 输入形状 {dummy_input.shape} -> 输出形状 {dummy_output.shape}")
print(f"  参数量: {sum(p.numel() for p in q_net_example.parameters()):,}")


In [ ]:
# ============================================================
# 经验回放缓冲区（Replay Buffer）
# ============================================================
# 从零实现：用 numpy 预分配固定大小的循环数组
# 效率比 Python list 高，适合频繁 push/sample

class ReplayBuffer:
    """
    循环经验回放缓冲区。

    存储 (s, a, r, s', done) transitions，支持均匀随机采样。

    Args:
        capacity: 最大容量
        state_dim: 状态维度
        device: 张量设备
    """
    def __init__(self, capacity: int, state_dim: int, device: str = 'cpu'):
        self.capacity = capacity
        self.device = torch.device(device)

        # 预分配 numpy 数组（连续内存，高效随机访问）
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)

        self._pos = 0       # 当前写入位置
        self._size = 0      # 当前有效数据量

    def push(self, state, action, reward, next_state, done, terminated):
        """
        存入一个 transition（覆盖最旧数据当容量满时）。
        """
        idx = self._pos % self.capacity
        self.states[idx] = state
        self.actions[idx] = action
        self.rewards[idx] = reward
        self.next_states[idx] = next_state
        self.dones[idx] = float(done)
        self.terminated[idx] = float(terminated)

        self._pos = (self._pos + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size: int):
        """
        从缓冲区中均匀随机采样一个 minibatch。

        Returns:
            (states, actions, rewards, next_states, dones, terminated)
            所有返回值为 torch.Tensor，位于 self.device 上。
        """
        indices = np.random.randint(0, self._size, size=batch_size)

        states = torch.FloatTensor(self.states[indices]).to(self.device)
        actions = torch.LongTensor(self.actions[indices]).to(self.device)
        rewards = torch.FloatTensor(self.rewards[indices]).to(self.device)
        next_states = torch.FloatTensor(self.next_states[indices]).to(self.device)
        dones = torch.FloatTensor(self.dones[indices]).to(self.device)
        terminated = torch.FloatTensor(self.terminated[indices]).to(self.device)

        return states, actions, rewards, next_states, dones, terminated

    def __len__(self):
        """当前存储的 transition 数量"""
        return self._size

    @property
    def is_ready(self, batch_size: int):
        """缓冲区是否已有足够数据开始训练"""
        return self._size >= batch_size


# 快速测试
buf = ReplayBuffer(capacity=100, state_dim=4)
buf.push(np.array([0.0, 0.0, 0.0, 0.0]), 0, 1.0, np.array([0.1, 0.0, 0.1, 0.0]), False, False)
print(f"缓冲区大小: {len(buf)} (期待 1)")
s, a, r, ns, d, term = buf.sample(1)
print(f"采样测试: state shape={s.shape}, action={a}, reward={r}")


In [ ]:
# ============================================================
# DQN 智能体（从零实现）
# ============================================================
# 本实现演示完整的 DQN 算法，包含：
# - 双网络结构（online Q + target Q）
# - eps-贪心探索（指数衰减）
# - 经验回放训练
# - 目标网络硬更新
# - 梯度裁剪
# - Huber 损失
#
# 与 rl_course.agents.dqn.DQNAgent 功能等价，
# 但这里用更教学化的方式逐步展示每个细节。

class DQNAgent:
    """
    深度 Q 网络智能体

    DQN 的核心思想：使用深度神经网络逼近 Q 函数，并通过
    经验回放和目标网络实现稳定训练。

    Args:
        state_dim: 状态空间维度
        n_actions: 离散动作数量
        hidden_dims: Q 网络隐藏层维度
        gamma: 折扣因子
        epsilon_start: 初始探索率
        epsilon_end: 最小探索率
        epsilon_decay: 探索率指数衰减系数（每次 act 后衰减）
        lr: 学习率
        batch_size: 训练批次大小
        buffer_capacity: 经验回放缓冲区容量
        target_update_freq: 目标网络更新频率（学习步数）
        device: 计算设备
        use_huber: 是否使用 Huber 损失
        max_grad_norm: 梯度裁剪的最大范数
    """

    def __init__(
        self,
        state_dim: int,
        n_actions: int,
        hidden_dims: list = None,
        gamma: float = 0.99,
        epsilon_start: float = 1.0,
        epsilon_end: float = 0.01,
        epsilon_decay: float = 0.998,
        lr: float = 1e-3,
        batch_size: int = 64,
        buffer_capacity: int = 10000,
        target_update_freq: int = 100,
        device: str = 'cpu',
        use_huber: bool = True,
        max_grad_norm: float = 10.0,
    ):
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.hidden_dims = hidden_dims or [64, 64]
        self.gamma = gamma
        self.epsilon = epsilon_start       # 当前探索率
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.lr = lr
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.device = torch.device(device)
        self.use_huber = use_huber
        self.max_grad_norm = max_grad_norm

        # ----- Q 网络（online）-----
        # 当前训练的 Q 函数，参数通过梯度下降实时更新
        # 用于：选择动作 + 计算当前 Q(s, a)
        self.q_network = QNetwork(
            state_dim=state_dim,
            n_actions=n_actions,
            hidden_dims=self.hidden_dims,
        ).to(self.device)

        # ----- 目标网络（target）-----
        # 用于计算 TD target: r + gamma * max Q_target(s', a')
        # 参数定期从 online 网络复制，保持稳定
        self.target_network = QNetwork(
            state_dim=state_dim,
            n_actions=n_actions,
            hidden_dims=self.hidden_dims,
        ).to(self.device)
        self._hard_update_target()  # 初始同步

        # ----- 优化器 -----
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)

        # ----- 损失函数 -----
        # 选择理由：
        # - Huber (SmoothL1): 大误差时梯度为常数，防止爆炸
        # - MSE: 大误差时梯度线性增长，可能不稳定
        if use_huber:
            self.criterion = nn.SmoothL1Loss()
        else:
            self.criterion = nn.MSELoss()

        # ----- 经验回放缓冲区 -----
        self.replay_buffer = ReplayBuffer(
            capacity=buffer_capacity,
            state_dim=state_dim,
            device=device,
        )

        # ----- 训练状态 -----
        self.learn_step_counter = 0     # 学习步数（用于触发 target 更新）
        self.total_steps = 0            # 总交互步数

        # ----- 训练指标记录 -----
        self.loss_history: List[float] = []         # 每一步的 loss
        self.q_value_history: List[float] = []      # 每一步的平均 Q 值
        self.epsilon_history: List[float] = []      # epsilon 变化

    def _hard_update_target(self):
        """
        硬复制：将 online 网络的所有参数复制给 target 网络。

        load_state_dict 是 PyTorch 中复制参数的标准方式。
        """
        self.target_network.load_state_dict(self.q_network.state_dict())

    def _preprocess_state(self, state):
        """
        将 numpy 状态转换为网络输入张量。

        Args:
            state: 形状 (state_dim,) 或 (batch_size, state_dim)

        Returns:
            float32 张量，形状 (1, state_dim) 或 (batch_size, state_dim)
        """
        if not isinstance(state, torch.Tensor):
            state = torch.FloatTensor(np.asarray(state, dtype=np.float32))
        if state.dim() == 1:
            state = state.unsqueeze(0)
        return state.to(self.device)

    # 其他方法在下面单元格定义...


In [ ]:
# ============================================================
# DQN 智能体：eps-贪心动作选择
# ============================================================
# 继续 DQNAgent 类的定义，此处添加 act 方法

def dqn_agent_act(self, obs, train=True):
    """
    eps-贪心策略选择动作。

    训练模式：
    - 以 eps 概率随机探索（uniform 采样）
    - 以 (1-eps) 概率利用（选择 Q 值最大的动作）
    - 每次调用后衰减 eps

    评估模式：
    - 始终选择 Q 值最大的动作（确定性策略）

    Args:
        obs: 观测状态，形状 (state_dim,)
        train: 是否为训练模式

    Returns:
        动作索引 int in [0, n_actions)
    """
    if train and np.random.random() < self.epsilon:
        # 探索：随机选择动作
        action = np.random.randint(0, self.n_actions)
    else:
        # 利用：选择当前 Q 值最大的动作
        state_tensor = self._preprocess_state(obs)  # (1, state_dim)
        with torch.no_grad():
            q_values = self.q_network(state_tensor)  # (1, n_actions)
            action = int(q_values.argmax(dim=1).item())

    # 训练模式下衰减探索率
    if train:
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        self.total_steps += 1

    return action

# 将方法绑定到类上
DQNAgent.act = dqn_agent_act


In [ ]:
# ============================================================
# DQN 智能体：TD 目标计算
# ============================================================

def dqn_agent_compute_target(self, rewards, next_states, terminated):
    """
    计算 DQN 的 TD 目标。

    标准 DQN 使用 target 网络的最大 Q 值：
        target = r + gamma * max_{a'} Q_target(s', a') * (1 - done)

    注意：
    - 使用目标网络（target_network），而非 online 网络
    - 取所有动作中的最大值（max 算子，也是过估计的根源）
    - done 状态的目标只包含即时奖励（未来回报为 0）

    Args:
        rewards: 即时奖励，形状 (batch_size,)
        next_states: 下一状态，形状 (batch_size, state_dim)
        dones: 终止标志（1=终止），形状 (batch_size,)

    Returns:
        TD 目标值，形状 (batch_size,)
    """
    with torch.no_grad():  # 目标网络不参与梯度计算
        # 目标网络预测下一状态的 Q 值
        next_q = self.target_network(next_states)  # (batch, n_actions)

        # 取最大值: max_{a'} Q_target(s', a')
        max_next_q = next_q.max(dim=1)[0]  # (batch,)

        # TD 目标公式
        # 当 terminated=1 时 (1-terminated)=0，目标 = r（终止后无未来回报）
        target = rewards + self.gamma * max_next_q * (1.0 - terminated)

    return target

DQNAgent._compute_target = dqn_agent_compute_target


In [ ]:
# ============================================================
# DQN 智能体：核心训练方法（update）
# ============================================================

def dqn_agent_update(self):
    """
    执行一次 DQN 参数更新。

    完整流程：
    1. 从经验回放中采样一个 minibatch
    2. 用 online 网络计算当前 Q(s, a)
    3. 用 target 网络计算 TD 目标
    4. 计算损失（Huber/MSE）
    5. 反向传播 + 梯度裁剪 + 优化器步进
    6. 定期硬更新 target 网络
    7. 记录训练指标

    Returns:
        {"loss": float, "epsilon": float, "avg_q": float, "grad_norm": float}
    """
    # 缓冲区数据不足时不更新
    if len(self.replay_buffer) < self.batch_size:
        return {"loss": 0.0, "epsilon": self.epsilon, "avg_q": 0.0, "grad_norm": 0.0}

    # --- 1. 采样 ---
    # 从历史经验中均匀随机采样
    states, actions, rewards, next_states, dones, terminated = \
        self.replay_buffer.sample(self.batch_size)

    # --- 2. 计算当前 Q(s, a) ---
    # Q(s, a): 网络输出所有动作的 Q 值，用 gather 选中实际动作
    q_values = self.q_network(states)                       # (batch, n_actions)
    q_value = q_values.gather(1, actions.unsqueeze(1))      # (batch, 1)
    q_value = q_value.squeeze(1)                             # (batch,)

    # --- 3. 计算 TD 目标 ---
    # 使用 target 网络（冻结参数），计算 r + gamma * max Q_target(s', a')
    target = self._compute_target(rewards, next_states, terminated)  # (batch,)

    # --- 4. 计算损失 ---
    # 公式: L = (r + gamma * max Q_target(s', a') - Q(s, a))^2
    # Huber Loss 替代 MSE 以提高对大误差的鲁棒性
    loss = self.criterion(q_value, target)

    # --- 5. 梯度下降 ---
    self.optimizer.zero_grad()
    loss.backward()

    # 梯度裁剪：将梯度的 L2 范数限制在 max_grad_norm 以内
    # 这是防止梯度爆炸的关键安全网
    grad_norm = torch.nn.utils.clip_grad_norm_(
        self.q_network.parameters(), self.max_grad_norm
    )

    self.optimizer.step()

    # --- 6. 记录指标 ---
    self.learn_step_counter += 1
    avg_q = q_value.mean().item()
    loss_val = loss.item()

    self.loss_history.append(loss_val)
    self.q_value_history.append(avg_q)

    # --- 7. 定期更新目标网络 ---
    # 每 target_update_freq 步将 online 网络参数复制给 target 网络
    if self.learn_step_counter % self.target_update_freq == 0:
        self._hard_update_target()

    return {
        "loss": loss_val,
        "epsilon": self.epsilon,
        "avg_q": avg_q,
        "grad_norm": grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm,
    }

DQNAgent.update = dqn_agent_update


In [ ]:
# ============================================================
# DQN 智能体：保存与加载
# ============================================================

def dqn_agent_save(self, path: str):
    """保存模型检查点"""
    checkpoint = {
        "q_network": self.q_network.state_dict(),
        "target_network": self.target_network.state_dict(),
        "optimizer": self.optimizer.state_dict(),
        "epsilon": self.epsilon,
        "learn_step_counter": self.learn_step_counter,
        "total_steps": self.total_steps,
        "config": {
            "state_dim": self.state_dim,
            "n_actions": self.n_actions,
            "hidden_dims": self.hidden_dims,
            "gamma": self.gamma,
            "epsilon_start": self.epsilon_start,
            "epsilon_end": self.epsilon_end,
            "epsilon_decay": self.epsilon_decay,
            "lr": self.lr,
            "batch_size": self.batch_size,
            "target_update_freq": self.target_update_freq,
            "use_huber": self.use_huber,
            "max_grad_norm": self.max_grad_norm,
        },
    }
    torch.save(checkpoint, path)
    print(f"模型已保存至: {path}")

def dqn_agent_load(self, path: str):
    """从检查点恢复模型"""
    checkpoint = torch.load(path, map_location=self.device)
    self.q_network.load_state_dict(checkpoint["q_network"])
    self.target_network.load_state_dict(checkpoint["target_network"])
    self.optimizer.load_state_dict(checkpoint["optimizer"])
    self.epsilon = checkpoint.get("epsilon", self.epsilon)
    self.learn_step_counter = checkpoint.get("learn_step_counter", 0)
    self.total_steps = checkpoint.get("total_steps", 0)
    print(f"模型已加载: {path}")

DQNAgent.save = dqn_agent_save
DQNAgent.load = dqn_agent_load


In [ ]:
import os
FAST_MODE = os.getenv('RL_COURSE_FAST_MODE', '0') == '1'
# ============================================================
# 训练配置：超参数详解
# ============================================================
# 所有超参数集中管理，便于实验对比

@dataclass
class DQNConfig:
    """
    DQN 超参数配置

    快速模式 (~5000 steps, 1-2 分钟 CPU):
    用于快速验证算法实现是否正确。

    完整模式 (~50000+ steps, 需要更多时间):
    用于获得收敛的 CartPole 智能体（可以达到 500 满分）。
    """
    # --- 环境相关 ---
    state_dim: int = 4          # CartPole 状态维度
    n_actions: int = 2          # 左推/右推

    # --- 网络结构 ---
    hidden_dims: tuple = (64, 64)  # 两个隐藏层，每层 64 个神经元
    # 64 对 CartPole 足够；Atari 游戏通常用更大的网络 (如 512)

    # --- RL 超参数 ---
    gamma: float = 0.99         # 折扣因子（越大越看重长远回报）
    # CartPole 是短视任务（每步+1），gamma=0.99 代表 100 步内 ~36.6% 的权重

    # --- 探索策略（eps-贪心）---
    epsilon_start: float = 1.0  # 从完全随机探索开始
    epsilon_end: float = 0.01   # 最小探索率（留少量随机性）
    epsilon_decay: float = 0.998 # 指数衰减系数

    # --- 优化 ---
    lr: float = 1e-3            # Adam 学习率
    batch_size: int = 32 if FAST_MODE else 64        # 每次更新的样本数
    # 更大的 batch 提供更稳定的梯度估计，但计算更慢

    # --- 经验回放 ---
    buffer_capacity: int = 2000 if FAST_MODE else 10000  # 最多存储 10000 条经验
    # CartPole 任务简单，10000 足够；Atari 常用 100000 ~ 1000000

    # --- 目标网络 ---
    target_update_freq: int = 100  # 每 100 步更新一次目标网络
    # 更新频率是稳定 vs 速度的折中

    # --- 损失与梯度 ---
    use_huber: bool = True       # Huber Loss vs MSE
    max_grad_norm: float = 10.0  # 梯度裁剪阈值

    # --- 训练控制 ---
    max_steps: int = 3000 if FAST_MODE else 20000  # FAST_MODE reduces steps
    warmup_steps: int = 500      # 预热：先填充缓冲区再训练
    # warmup 确保第一次更新时缓冲区有足够的多样数据

    # --- 设备 ---
    device: str = 'cpu'

# 创建配置
config = DQNConfig()
print("DQN 超参数配置:")
for key in config.__dataclass_fields__:
    print(f"  {key}: {getattr(config, key)}")


In [ ]:
# ============================================================
# 训练循环
# ============================================================
# 核心训练流程：
# 1. 重置环境，获取初始状态
# 2. 循环交互：act -> step -> push -> update
# 3. 记录每个 episode 的回报
# 4. 定期输出训练状态

def train_dqn(config):
    """
    DQN 训练主函数。

    Args:
        config: 超参数配置

    Returns:
        (agent, episode_rewards, episode_lengths)
    """
    # 创建环境（每次训练新建，避免状态污染）
    env = gym.make('CartPole-v1')

    # 创建智能体
    agent = DQNAgent(
        state_dim=config.state_dim,
        n_actions=config.n_actions,
        hidden_dims=list(config.hidden_dims),
        gamma=config.gamma,
        epsilon_start=config.epsilon_start,
        epsilon_end=config.epsilon_end,
        epsilon_decay=config.epsilon_decay,
        lr=config.lr,
        batch_size=config.batch_size,
        buffer_capacity=config.buffer_capacity,
        target_update_freq=config.target_update_freq,
        device=config.device,
        use_huber=config.use_huber,
        max_grad_norm=config.max_grad_norm,
    )

    episode_rewards: List[float] = []   # 每个 episode 的总回报
    episode_lengths: List[int] = []     # 每个 episode 的步数
    recent_rewards = deque(maxlen=20)   # 最近 20 个 episode 的平均回报（用于打印）
    best_mean_reward = -float('inf')

    # 交互与训练
    obs, _ = env.reset(seed=42)
    episode_reward = 0.0
    episode_step = 0
    episode_num = 0

    pbar = tqdm(total=config.max_steps, desc="DQN Training", position=0)

    for step in range(1, config.max_steps + 1):
        # --- 动作选择 ---
        action = agent.act(obs, train=True)

        # --- 环境交互 ---
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # --- 存储经验 ---
        agent.replay_buffer.push(obs, action, reward, next_obs, done, terminated)

        # --- 经验回放训练 ---
        # 只有缓冲区数据足够时才更新
        if step >= config.warmup_steps:
            update_info = agent.update()
        else:
            update_info = {"loss": 0.0, "epsilon": agent.epsilon, "avg_q": 0.0}

        # --- 记录 ---
        episode_reward += reward
        episode_step += 1
        obs = next_obs

        # --- 记录探索率 ---
        agent.epsilon_history.append(agent.epsilon)

        # --- episode 结束处理 ---
        if done:
            episode_rewards.append(episode_reward)
            episode_lengths.append(episode_step)
            recent_rewards.append(episode_reward)
            episode_num += 1

            # 更新最佳模型
            mean_reward = np.mean(recent_rewards) if recent_rewards else 0.0
            if mean_reward > best_mean_reward and step >= config.warmup_steps:
                best_mean_reward = mean_reward
                agent.save('outputs/checkpoints/dqn_cartpole_best.pt')

            # 进度显示
            pbar.set_postfix({
                'ep': episode_num,
                'R': f'{episode_reward:.1f}',
                'avgR': f'{np.mean(recent_rewards):.1f}' if recent_rewards else 'N/A',
                'eps': f'{agent.epsilon:.3f}',
                'loss': f'{update_info["loss"]:.3f}',
            })

            # 重置环境
            obs, _ = env.reset()
            episode_reward = 0.0
            episode_step = 0

        pbar.update(1)

    pbar.close()
    env.close()

    # 保存最终模型
    agent.save('outputs/checkpoints/dqn_cartpole.pt')
    print(f"\n训练完成！共 {episode_num} 个 episodes, {config.max_steps} 步")
    print(f"   最终平均回报 (最近20ep): {np.mean(recent_rewards):.2f} +/- {np.std(recent_rewards):.2f}")
    print(f"   最佳平均回报: {best_mean_reward:.2f}")

    return agent, episode_rewards, episode_lengths


# ===== 开始训练 =====
print("=" * 60)
print("开始 DQN 训练（快速模式，约 5000 步，1-2 分钟）")
print("=" * 60)

agent, rewards, lengths = train_dqn(config)

print(f"\n奖励统计: 均值={np.mean(rewards):.1f}, 最大值={max(rewards):.1f}")
print(f"步数统计: 均值={np.mean(lengths):.1f}, 最大值={max(lengths):.1f}")


In [ ]:
# ============================================================
# 绘制训练曲线：Episode 回报随时间变化
# ============================================================
# 训练曲线是评判算法收敛性的核心指标：
# - 上升趋势 -> 智能体在学习
# - 大幅震荡 -> 不稳定
# - 平直且低 -> 未学到有效策略

def plot_training_results(rewards, lengths, config):
    """绘制训练结果：回报曲线和步数曲线"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    # --- 左图：Episode 回报 ---
    ax = axes[0]
    ax.plot(rewards, alpha=0.3, linewidth=0.7, color='steelblue', label='Raw Reward')
    # 滑动平均
    window = min(10, len(rewards) // 2) if len(rewards) > 10 else 1
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed,
                linewidth=2, color='darkblue', label=f'Smoothed (w={window})')
    ax.axhline(y=500, color='gray', linestyle='--', alpha=0.5, label='Max (500)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.set_title(f'DQN 训练曲线 (Total Steps: {config.max_steps})')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- 右图：Episode 长度 ---
    ax = axes[1]
    ax.plot(lengths, alpha=0.3, linewidth=0.7, color='coral', label='Raw Length')
    if len(lengths) >= window:
        smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(lengths)), smoothed,
                linewidth=2, color='darkred', label=f'Smoothed (w={window})')
    ax.axhline(y=500, color='gray', linestyle='--', alpha=0.5, label='Max (500)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Length')
    ax.set_title('Episode Length over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig('outputs/figures/10_dqn_cartpole.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("训练曲线已保存至 outputs/figures/10_dqn_cartpole.png")

plot_training_results(rewards, lengths, config)


In [ ]:
# ============================================================
# 评估训练好的智能体（确定性策略，无探索）
# ============================================================
# 评估模式：epsilon=0，始终选择 Q 值最大的动作
# 运行 10 个 episodes 取平均

def evaluate_agent(agent, n_episodes=10):
    """
    评估 DQN 智能体表现。

    Args:
        agent: 训练好的 DQNAgent
        n_episodes: 评估的 episode 数量

    Returns:
        (mean_return, std_return)
    """
    env = gym.make('CartPole-v1')
    eval_rewards = []

    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0.0
        step = 0

        while not done and step < 500:
            # 评估模式：确定性的贪心策略
            action = agent.act(obs, train=False)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            step += 1

        eval_rewards.append(total_reward)
        print(f"  评估 Episode {ep+1}: 回报={total_reward:.1f}, 步数={step}")

    env.close()

    mean_r = np.mean(eval_rewards)
    std_r = np.std(eval_rewards)
    print(f"\n评估结果 ({n_episodes} episodes):")
    print(f"   平均回报: {mean_r:.1f} +/- {std_r:.1f}")
    print(f"   最大回报: {max(eval_rewards):.1f}")

    return mean_r, std_r

mean_r, std_r = evaluate_agent(agent, n_episodes=10)


In [ ]:
# ============================================================
# 录制智能体玩 CartPole 的视频
# ============================================================
# 使用 rl_course.visualization.video.record_episode
# 通过 env.render("rgb_array") 逐帧采集，imageio 合成 GIF/MP4

# 定义策略函数（评估模式，无探索）
def policy_fn(state):
    return agent.act(state, train=False)

# 创建用于录制的环境（需支持 rgb_array 渲染）
video_env = gym.make('CartPole-v1', render_mode='rgb_array')

# 录制 GIF
video_path = record_episode(
    env=video_env,
    policy_fn=policy_fn,
    filepath='outputs/videos/10_dqn_cartpole.gif',
    fps=30,
    max_steps=500,
    format='gif',
    verbose=False,
)

video_env.close()

# 在 notebook 中显示
print(f"视频已保存: {video_path}")
try:
    display(Image(open(video_path, 'rb').read(), format='png'))
except Exception:
    print("(GIF 在纯文本环境中无法内嵌显示)")


In [ ]:
# ============================================================
# 分析一：Q 值变化趋势
# ============================================================
# Q 值反映了智能体对状态-动作对的估计。
# 训练过程中，Q 值应逐渐趋向于真实期望回报。

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# --- 左图：平均 Q 值变化 ---
ax = axes[0]
ax.plot(agent.q_value_history, linewidth=0.7, color='green', alpha=0.5, label='Avg Q per update')
if len(agent.q_value_history) > 20:
    smoothed = np.convolve(agent.q_value_history, np.ones(20)/20, mode='valid')
    ax.plot(range(19, len(agent.q_value_history)), smoothed,
            linewidth=2, color='darkgreen', label='Smoothed (w=20)')
ax.set_xlabel('Update Step')
ax.set_ylabel('Average Q(s, a)')
ax.set_title('Q Value During Training')
ax.legend()
ax.grid(True, alpha=0.3)

# --- 右图：最终 Q 值分布（随机采样状态）---
ax = axes[1]
# 从缓冲区取 100 个状态
sample_states = agent.replay_buffer.states[:min(100, len(agent.replay_buffer))]
if len(sample_states) > 0:
    states_tensor = torch.FloatTensor(sample_states).to(agent.device)
    with torch.no_grad():
        q_vals = agent.q_network(states_tensor).cpu().numpy()  # (N, 2)
    ax.hist(q_vals[:, 0], alpha=0.6, bins=20, label='Q(s, left)', color='steelblue')
    ax.hist(q_vals[:, 1], alpha=0.6, bins=20, label='Q(s, right)', color='coral')
    ax.set_xlabel('Q Value')
    ax.set_ylabel('Count')
    ax.set_title('Q Value Distribution (after training)')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No data in buffer', ha='center', va='center', transform=ax.transAxes)

fig.tight_layout()
fig.savefig('outputs/figures/10_dqn_q_values.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# 分析二：Epsilon 衰减曲线
# ============================================================
# eps-贪心策略在训练初期以高概率探索（学到更多），
# 后期以低概率利用（表现最优策略）。
# 理想的 eps 衰减应在探索和利用之间平衡。

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# --- 左图：Epsilon 随时间变化 ---
ax = axes[0]
eps_history = agent.epsilon_history
ax.plot(eps_history, linewidth=1.5, color='purple')
ax.axhline(y=config.epsilon_end, color='red', linestyle='--', alpha=0.7, label=f'eps_end={config.epsilon_end}')
ax.axvline(x=config.warmup_steps, color='gray', linestyle=':', alpha=0.7, label=f'warmup={config.warmup_steps}')
ax.set_xlabel('Step')
ax.set_ylabel('Epsilon')
ax.set_title('Epsilon Decay Schedule')
ax.legend()
ax.grid(True, alpha=0.3)

# --- 右图：理论衰减曲线 ---
ax = axes[1]
steps_ar = np.arange(0, config.max_steps)
theoretical_eps = config.epsilon_end + (config.epsilon_start - config.epsilon_end) * \
    (config.epsilon_decay ** steps_ar)
ax.plot(steps_ar, theoretical_eps, linewidth=2, color='darkorange', label=f'decay={config.epsilon_decay}')
ax.axhline(y=config.epsilon_end, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Step')
ax.set_ylabel('Epsilon')
ax.set_title('Theoretical Epsilon Decay')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

fig.tight_layout()
fig.savefig('outputs/figures/10_dqn_epsilon.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"最终 epsilon: {agent.epsilon:.4f}")
print(f"总步数: {agent.total_steps}")
print(f"理论 epsilon 在 {config.max_steps} 步后: {theoretical_eps[-1]:.4f}")


In [ ]:
# ============================================================
# 分析三：损失曲线与梯度分析
# ============================================================
# 损失曲线反映 Q 网络对 TD target 的拟合程度。
# 正常情况：loss 在训练初期较高（Q 值估计不准），
# 随着训练推进逐渐下降并稳定。

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# --- 左图：损失变化 ---
ax = axes[0]
losses = agent.loss_history
ax.plot(losses, linewidth=0.5, alpha=0.4, color='red', label='Raw Loss')
if len(losses) > 20:
    smoothed = np.convolve(losses, np.ones(20)/20, mode='valid')
    ax.plot(range(19, len(losses)), smoothed, linewidth=2, color='darkred', label='Smoothed (w=20)')
ax.set_xlabel('Update Step')
ax.set_ylabel('Loss')
ax.set_title('DQN Loss (Huber) over Training')
ax.legend()
ax.grid(True, alpha=0.3)

# --- 右图：Q 值 vs Loss 散点图 ---
ax = axes[1]
if len(agent.q_value_history) > 0 and len(losses) > 0:
    # 截取相同长度
    min_len = min(len(agent.q_value_history), len(losses))
    qs = agent.q_value_history[:min_len]
    ls = losses[:min_len]
    ax.scatter(qs, ls, s=1, alpha=0.3, c='steelblue')
    ax.set_xlabel('Average Q(s, a)')
    ax.set_ylabel('Loss')
    ax.set_title('Q Value vs Loss')
    ax.grid(True, alpha=0.3)

    # 趋势线
    z = np.polyfit(qs, ls, 1)
    p = np.poly1d(z)
    x_line = np.linspace(min(qs), max(qs), 50)
    ax.plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Trend')
    ax.legend()

fig.tight_layout()
fig.savefig('outputs/figures/10_dqn_loss.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Loss 统计:")
if losses:
    print(f"  最小值: {min(losses):.4f}")
    print(f"  最终值: {losses[-1]:.4f}")
if agent.q_value_history:
    print(f"  Q 值范围: {min(agent.q_value_history):.2f} ~ {max(agent.q_value_history):.2f}")



## 6. DQN 常见的失败模式

即使有了三大创新，DQN 在训练中仍然可能出现各种问题。理解这些失败模式对于调试至关重要。

### 6.1 无经验回放：灾难性遗忘

**现象**：智能体刚学会保持杆子直立，很快就忘记了，表现剧烈波动。

**原因**：连续 transitions 高度相关，每次更新都在"加强"当前策略的偏好，形成正反馈循环。
当策略发生偏移后，之前学到的知识被快速覆盖（灾难性遗忘）。

**可视化类比**：
- 有回放：像学生复习错题本，随机抽取不同时期的题目
- 无回放：像学生只做最新的一道题，反复练习直到背下答案，但换道新题就不会了

### 6.2 无目标网络：Q 值爆炸

**现象**：Loss 持续上升最终发散到 NaN，Q 值增长到数百万。

**原因**：TD target 和预测 Q 值使用同一组参数，每次更新**同时**改变了目标和预测。
当 target 跟随预测变化时，可能形成正反馈：

```
Q(s,a) 被高估 -> y = r + gamma * max Q(s') 也被高估
-> 梯度要求 Q(s,a) 更高 -> Q(s,a) 再次升高
-> y 再次升高 -> ... 不断循环
```

### 6.3 Overestimation Bias（过估计偏差）

**现象**：Q 值显著高于真实期望回报（例如 CartPole 中 Q 值达到 1000+，但真实回报上限 500）。

**原因**：标准 DQN 使用 max_{a'} Q(s', a') 计算 target。由于 Q 函数存在估计误差，
最大值算子会**放大正向误差**：

$$E[max(X_1, X_2)] >= max(E[X_1], E[X_2])$$

**解决方案**：Double DQN（第 11 节详述）将动作选择和动作评估解耦。

### 6.4 其他常见问题

| 问题 | 症状 | 可能的解决方案 |
|------|------|-----------------|
| 学习率太高 | 训练振荡 | 降低 lr 到 3e-4 或 1e-4 |
| 缓冲区太小 | 过拟合近期经验 | 增加 buffer_capacity |
| Target 更新太频繁 | 目标不稳定 | 增大 target_update_freq |
| Epsilon 衰减太快 | 过早收敛到次优策略 | 降低 epsilon_decay |
| 网络太浅 | 无法拟合复杂 Q 函数 | 增加层数或神经元数 |
| 梯度爆炸 | loss 突然增大到 NaN | 降低 lr, 增加梯度裁剪 |


In [ ]:
# ============================================================
# 消融实验一：移除目标网络
# ============================================================
# 对比实验：使用相同的超参数，但在更新时使用 online 网络
# 计算 TD target（即不冻结目标）。
#
# 预期结果：训练不稳定，Q 值可能发散或剧烈震荡。

def train_dqn_no_target(config):
    """
    没有目标网络的 DQN（在线 Q-learning + 神经网络）。
    每次更新使用同一网络计算 TD target：
        y = r + gamma * max Q(s', a'; theta)  <- 用 theta 而非 theta-
    """
    env = gym.make('CartPole-v1')
    agent = DQNAgent(
        state_dim=config.state_dim,
        n_actions=config.n_actions,
        hidden_dims=list(config.hidden_dims),
        gamma=config.gamma,
        epsilon_start=config.epsilon_start,
        epsilon_end=config.epsilon_end,
        epsilon_decay=config.epsilon_decay,
        lr=config.lr,
        batch_size=config.batch_size,
        buffer_capacity=config.buffer_capacity,
        target_update_freq=config.target_update_freq,
        device=config.device,
        use_huber=config.use_huber,
        max_grad_norm=config.max_grad_norm,
    )
    episode_rewards = []
    obs, _ = env.reset(seed=42)
    episode_reward = 0
    episode_step = 0
    episode_num = 0

    pbar = tqdm(total=config.max_steps, desc="No TargetNet", position=0)

    for step in range(1, config.max_steps + 1):
        action = agent.act(obs, train=True)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.replay_buffer.push(obs, action, reward, next_obs, done, terminated)

        if step >= config.warmup_steps and len(agent.replay_buffer) >= agent.batch_size:
            # ----- 关键区别：使用 online 网络而非 target 网络 -----
            states, actions, rewards_b, next_states, dones, terminated = agent.replay_buffer.sample(agent.batch_size)

            q_values = agent.q_network(states)
            q_value = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

            # 直接使用 online 网络计算 max Q(s', a')（无目标网络！）
            with torch.no_grad():
                next_q = agent.q_network(next_states)  # 使用同一个 q_network！
                max_next_q = next_q.max(dim=1)[0]
                target = rewards_b + agent.gamma * max_next_q * (1.0 - terminated)

            loss = agent.criterion(q_value, target)
            agent.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.q_network.parameters(), agent.max_grad_norm)
            agent.optimizer.step()

        episode_reward += reward
        episode_step += 1
        obs = next_obs

        if done:
            episode_rewards.append(episode_reward)
            episode_num += 1
            pbar.set_postfix({'ep': episode_num, 'R': f'{episode_reward:.1f}', 'eps': f'{agent.epsilon:.3f}'})
            obs, _ = env.reset()
            episode_reward = 0
            episode_step = 0

        pbar.update(1)

    pbar.close()
    env.close()
    return episode_rewards, agent

print("=" * 60)
print("消融实验：移除目标网络")
print("=" * 60)
rewards_no_target, agent_no_target = train_dqn_no_target(config)
print(f"\n完成！共 {len(rewards_no_target)} episodes")


In [ ]:
# ============================================================
# 消融实验二：移除经验回放
# ============================================================
# 对比实验：不使用经验回放缓冲区，每次用最新的 transition
# 进行在线更新（SGD on each step）。
#
# 预期结果：灾难性遗忘，训练曲线剧烈震荡或无法学习。

def train_dqn_no_replay(config):
    """
    没有经验回放的 DQN（在线 SGD）。
    每次只用最新的 transition 更新网络：
        - 不存储经验
        - 每步用当前 (s, a, r, s', done) 直接更新
        - 不满足 i.i.d. 假设，梯度估计有偏
    """
    env = gym.make('CartPole-v1')
    agent = DQNAgent(
        state_dim=config.state_dim,
        n_actions=config.n_actions,
        hidden_dims=list(config.hidden_dims),
        gamma=config.gamma,
        epsilon_start=config.epsilon_start,
        epsilon_end=config.epsilon_end,
        epsilon_decay=config.epsilon_decay,
        lr=config.lr,
        batch_size=config.batch_size,
        buffer_capacity=config.buffer_capacity,
        target_update_freq=config.target_update_freq,
        device=config.device,
        use_huber=config.use_huber,
        max_grad_norm=config.max_grad_norm,
    )
    episode_rewards = []
    obs, _ = env.reset(seed=42)
    episode_reward = 0
    episode_step = 0
    episode_num = 0

    pbar = tqdm(total=config.max_steps, desc="No Replay", position=0)

    for step in range(1, config.max_steps + 1):
        action = agent.act(obs, train=True)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # ----- 关键区别：用当前 transition 直接更新（无采样）-----
        if step >= config.warmup_steps:
            # 将单个 transition 转为 batch 形式
            s = torch.FloatTensor(np.array([obs], dtype=np.float32)).to(agent.device)
            a = torch.LongTensor([action]).to(agent.device)
            r = torch.FloatTensor([reward]).to(agent.device)
            ns = torch.FloatTensor(np.array([next_obs], dtype=np.float32)).to(agent.device)
            d = torch.FloatTensor([float(done)]).to(agent.device)

            q_values = agent.q_network(s)
            q_value = q_values.gather(1, a.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                next_q = agent.target_network(ns)
                max_next_q = next_q.max(dim=1)[0]
                target = r + agent.gamma * max_next_q * (1.0 - d)

            loss = agent.criterion(q_value, target)
            agent.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.q_network.parameters(), agent.max_grad_norm)
            agent.optimizer.step()

        episode_reward += reward
        episode_step += 1
        obs = next_obs

        if done:
            episode_rewards.append(episode_reward)
            episode_num += 1
            pbar.set_postfix({'ep': episode_num, 'R': f'{episode_reward:.1f}', 'eps': f'{agent.epsilon:.3f}'})
            obs, _ = env.reset()
            episode_reward = 0
            episode_step = 0

        pbar.update(1)

    pbar.close()
    env.close()
    return episode_rewards, agent

print("=" * 60)
print("消融实验：移除经验回放")
print("=" * 60)
rewards_no_replay, agent_no_replay = train_dqn_no_replay(config)
print(f"\n完成！共 {len(rewards_no_replay)} episodes")


In [ ]:
# ============================================================
# 消融实验对比可视化
# ============================================================
# 将三种配置的学习曲线放在同一张图中对比：
# 1. 完整 DQN（带 target net + replay buffer）
# 2. DQN - target network（没有目标网络）
# 3. DQN - experience replay（没有经验回放）

def moving_average(data, window=10):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- 左图：原始回报 ---
ax = axes[0]

# 完整 DQN
ax.plot(rewards, alpha=0.3, linewidth=0.5, color='darkblue')
if len(rewards) > 10:
    ax.plot(range(9, len(rewards)), moving_average(rewards, 10),
            linewidth=2, color='darkblue', label='Full DQN')

# No Target
ax.plot(rewards_no_target, alpha=0.3, linewidth=0.5, color='darkred')
if len(rewards_no_target) > 10:
    ax.plot(range(9, len(rewards_no_target)), moving_average(rewards_no_target, 10),
            linewidth=2, color='darkred', label='No Target Network')

# No Replay
ax.plot(rewards_no_replay, alpha=0.3, linewidth=0.5, color='darkgreen')
if len(rewards_no_replay) > 10:
    ax.plot(range(9, len(rewards_no_replay)), moving_average(rewards_no_replay, 10),
            linewidth=2, color='darkgreen', label='No Experience Replay')

ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('DQN Ablation Study: Episode Rewards')
ax.legend()
ax.grid(True, alpha=0.3)

# --- 右图：统计对比柱状图 ---
ax = axes[1]
labels = ['Full DQN', 'No Target\nNetwork', 'No Experience\nReplay']
means = [
    np.mean(rewards) if rewards else 0,
    np.mean(rewards_no_target) if rewards_no_target else 0,
    np.mean(rewards_no_replay) if rewards_no_replay else 0,
]
stds = [
    np.std(rewards) if rewards else 0,
    np.std(rewards_no_target) if rewards_no_target else 0,
    np.std(rewards_no_replay) if rewards_no_replay else 0,
]
maxes = [
    max(rewards) if rewards else 0,
    max(rewards_no_target) if rewards_no_target else 0,
    max(rewards_no_replay) if rewards_no_replay else 0,
]

x = np.arange(len(labels))
width = 0.25
ax.bar(x - width, means, width, yerr=stds, capsize=3, label='Mean Reward', color='steelblue')
ax.bar(x, maxes, width, label='Max Reward', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Reward')
ax.set_title('Ablation Study: Performance Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

fig.tight_layout()
fig.savefig('outputs/figures/10_dqn_ablation.png', dpi=120, bbox_inches='tight')
plt.show()

print("消融实验对比图已保存至 outputs/figures/10_dqn_ablation.png")
print()
print("消融实验统计:")
print(f"{'配置':<25} {'Episodes':<10} {'平均回报':<12} {'最大回报':<12}")
print("-" * 60)
print(f"{'Full DQN':<25} {len(rewards):<10} {np.mean(rewards):<12.1f} {max(rewards):<12.1f}")
print(f"{'No Target Network':<25} {len(rewards_no_target):<10} {np.mean(rewards_no_target):<12.1f} {max(rewards_no_target):<12.1f}")
print(f"{'No Experience Replay':<25} {len(rewards_no_replay):<10} {np.mean(rewards_no_replay):<12.1f} {max(rewards_no_replay):<12.1f}")



## 7. 消融实验分析

### 结果解读

| 配置 | 预期表现 | 实际观察 |
|------|----------|----------|
| **完整 DQN** | 稳定学习，回报逐步上升 | 应有稳定的上升曲线 |
| **无 Target Network** | 训练不稳定，Q 值可能发散 | 曲线震荡剧烈，可能永远达不到高分 |
| **无 Experience Replay** | 灾难性遗忘，无法保持学到的知识 | 早期可能短暂学到一些，但很快遗忘 |

### 为什么 Target Network 这么重要？

没有目标网络时，TD target 和当前预测使用同一组参数。考虑以下情形：

1. Q 网络偶然高估了某个动作的 Q 值
2. TD target y = r + gamma * max Q(s') 同样被高估
3. 损失函数要求 Q(s, a) 向这个偏高的 target 靠近
4. Q(s, a) 被推得更高
5. 下一轮，target 再次被抬高 ...
6. **正反馈循环**导致 Q 值发散

### 为什么 Experience Replay 这么重要？

没有经验回放时，连续 transitions 高度相关。考虑以下情形：

1. 智能体连续几次都选择了向左推
2. 网络开始"过度适应"向左动作对应的状态
3. 当策略需要转向向右推时，网络保留了大量向左的"记忆"
4. 新知识快速覆盖旧知识，导致**灾难性遗忘**

### 关于 Double DQN

标准 DQN 使用 max_{a'} Q(s', a') 作为 target，这导致系统性的**过估计偏差**。
Double DQN 的改进很简单：使用 online 网络选择动作，target 网络评估值：

$$a* = \arg\max_{a'} Q_{online}(s', a')$$
$$y = r + \gamma * Q_{target}(s', a*)$$

这将在第 11 节中详细实现。



## 8. 本节总结

### 核心知识点

| 概念 | 说明 | 重要性 |
|------|------|--------|
| **DQN 三大创新** | 经验回放、目标网络、Huber Loss | *** |
| **经验回放** | 打破数据相关，提高样本效率 | *** |
| **目标网络** | 固定 TD target，稳定训练 | *** |
| **eps-贪心探索** | 平衡探索与利用 | ** |
| **梯度裁剪** | 防止梯度爆炸 | ** |
| **Huber Loss** | 对大误差更鲁棒 | ** |
| **Overestimation** | max 算子导致 Q 值高估 | ** |

### DQN 损失函数

$$L(theta) = (1/N) * sum_{i=1}^{N} Huber( r_i + gamma * max_{a'} Q_target(s'_i, a'; theta^-) * (1-d_i) - Q(s_i, a_i; theta) )$$

### DQN vs Tabular Q-Learning

| 方面 | Tabular Q-Learning | DQN |
|------|-------------------|-----|
| Q 函数表示 | 查表（每个 (s,a) 独立） | 神经网络（参数共享） |
| 泛化能力 | 无（未访问状态不可知） | 强（相似状态有相似 Q 值） |
| 高维状态 | 不可行（维数灾难） | 可行（图像、传感器等） |
| 收敛保证 | 有（有限 MDP + 合适超参） | 无（非线性函数逼近） |
| 训练稳定性 | 较稳定 | 需要多种技巧稳定 |

### 下一步

- **[11_double_dqn.ipynb]** - 用 Double DQN 解决过估计问题
- **[12_reinforce.ipynb]** - 切换到策略梯度方法
- **[14_actor_critic.ipynb]** - Actor-Critic：结合价值学习和策略学习


## 9. 面试题（点击展开）

<details>
<summary><b>点击查看 DQN 相关面试问题</b></summary>

---

### 基础问题

**Q1: DQN 的三大核心创新是什么？为什么需要它们？**

A: (1) 经验回放 - 打破数据相关性；(2) 目标网络 - 稳定 TD target；(3) Reward Clipping / Huber Loss - 控制梯度尺度。

**Q2: 为什么 DQN 需要经验回放？在线学习有什么问题？**

A: 在线学习中连续 transition 高度相关，不满足 i.i.d. 假设，导致梯度方差大、参数震荡。经验回放通过均匀采样打破相关性。

**Q3: 目标网络解决了什么问题？如果不用会怎样？**

A: 解决目标非平稳问题。不用时 TD target 随参数变化，形成正反馈导致 Q 值发散。

### 进阶问题

**Q4: 解释 DQN 的 overestimation bias 产生原因。**

A: max 算子放大了正向估计误差：E[max(Q1, Q2)] >= max(E[Q1], E[Q2])。

**Q5: Huber Loss 相比 MSE 有什么优势？**

A: MSE 在大误差时梯度线性增长，易梯度爆炸。Huber Loss 在大误差时梯度为常数，更鲁棒。

**Q6: 如何选择目标网络的更新频率？**

A: 太频繁（如每步更新）-> 退化为无目标网络；太低（如 10000 步）-> target 太陈旧。CartPole 常用 100，Atari 常用 10000。

**Q7: 梯度裁剪在 DQN 中的作用是什么？**

A: 将梯度 L2 范数限制在阈值内，防止极端 TD error 导致的梯度爆炸。

### 深入问题

**Q8: DQN 是否保证收敛？为什么？**

A: 不保证。DQN 使用非线性函数逼近（神经网络），打破了 tabular Q-Learning 的收敛条件。实践中通过经验回放和目标网络获得近似收敛。

**Q9: 经验回放缓冲区的大小如何影响学习？**

A: 太小 -> 样本单一，过拟合近期经验；太大 -> 包含过多旧数据（策略已改变），学习效率低。

**Q10: CartPole 中 DQN 成功的关键是什么？**

A: 低维状态（4 维）易于拟合；奖励稠密（每步+1）；动作空间小（2 个）；任务相对简单。

---
</details>



## 10. 练习

### 基础练习

1. **调整超参数**：修改 epsilon_decay 为 0.99 和 0.9999，观察训练曲线的变化。过快的衰减会导致什么？过慢呢？

2. **替换损失函数**：将 use_huber 改为 False（使用 MSE Loss），比较训练过程的差异。MSE 是否更容易发散？

3. **调整网络结构**：将隐藏层从 [64, 64] 改为 [16]（单层小网络）和 [256, 256]（更大网络），观察学习能力的差异。

### 进阶练习

4. **实现优先级采样**：在 ReplayBuffer 中按 TD error 的大小进行优先级采样（Prioritized Experience Replay），而不是均匀采样。比较训练效率。

5. **实现 Soft Update**：将目标网络的硬更新（hard update）改为软更新（soft update）：
   theta- <- tau * theta + (1 - tau) * theta-
   尝试不同的 tau 值（如 0.01, 0.001）。

6. **探索与利用平衡**：实现一个线性衰减的 epsilon 调度器（而非指数衰减），比较两者的效果差异。

### 挑战练习

7. **Adapt to LunarLander**：将 DQN 应用到 LunarLander-v3 环境（gymnasium 中）。这个环境有 8 维状态和 4 个离散动作。需要调整哪些超参数？训练是否更困难？

8. **实现 Double DQN**：修改 _compute_target 方法，使用 online 网络选择动作、target 网络评估值，对比标准 DQN 和 Double DQN 的 Q 值估计。

9. **Dueling DQN**：将 Q 网络分解为 V(s) + A(s, a)（状态价值 + 优势函数），实现 Dueling DQN 架构。在 CartPole 上比较与标准 DQN 的性能差异。
